In [4]:
%pip install -e /home/jag/model-engine

Looking in indexes: http://zamlpkgs-nginx/artifactory/api/pypi/zest_pypi/simple/
Obtaining file:///home/jag/model-engine
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for model_engine (pyproject.toml) ... done
  Created wheel for model_engine: filename=model_engine-2.1.4-0.editable-py3-none-any.whl size=3500 sha256=60a11ecdc19fa17e9f9fc17970e2665d9d63f54d119a54d4bb4530168e82c541
  Stored in directory: /tmp/pip-ephem-wheel-cache-97vb19yq/wheels/82/45/c3/5719ce6688788a799da0a8a1c1d462af250e1118c85f1ce765
Successfully built model_engine
  Attempting uninstall: model_engine
    Found existing installation: model_engine 2.1.4
    Uninstalling model_engine-2.1.4:
      Successfully uninstalled model_engine-2.1.4

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /home/jag/.cond

In [5]:
%pip install -e /home/jag/feature-engine-parts-processor-change

Looking in indexes: http://zamlpkgs-nginx/artifactory/api/pypi/zest_pypi/simple/
Obtaining file:///home/jag/feature-engine-parts-processor-change
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for feature-engine-parts (pyproject.toml) ... done
  Created wheel for feature-engine-parts: filename=feature_engine_parts-2.0.2-0.editable-py3-none-any.whl size=21908 sha256=8749024e040cad66ebf0d8808b63f4b03440b35455fbbb721ec27050b0645563
  Stored in directory: /tmp/pip-ephem-wheel-cache-gycvxsrd/wheels/62/df/f6/b3bea7055bbbe8f5cf4259075d5ca3acceff669bda2d53ab18
Successfully built feature-engine-parts
  Attempting uninstall: feature-engine-parts
    Found existing installation: feature-engine-parts 2.0.2
    Uninstalling feature-engine-parts-2.0.2:
      Successfully uninstalled feature-engine-parts-2.0.2

[notice] A 

In [6]:
pip show model-engine

Name: model_engine
Version: 2.1.4
Summary: Zest model building engine
Home-page: http://github.com/Katlean/model-engine
Author: Zest Finance Data Science Team
Author-email: core-modeling@zestfinance.com
License: 
Location: /home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages
Editable project location: /home/jag/model-engine
Requires: agparser, feature-engine-parts, fsspec, numpy, packaging, pandas, s3fs, zaml, zamlexplain, zestio
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip show feature-engine-parts

Name: feature-engine-parts
Version: 2.0.2
Summary: Feature Engine Parts
Home-page: https://github.com/Katlean/feature-engine-parts
Author: Zest AI Data Science Team
Author-email: core-modeling@zestfinance.com
License: 
Location: /home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages
Editable project location: /home/jag/feature-engine-parts-processor-change
Requires: dill, numpy, pandas, revops_tools_s3, scikit-learn, xgboost, zamlmodel
Required-by: model_engine
Note: you may need to restart the kernel to use updated packages.


In [8]:
from feature_engine_parts.fe_parts_V2.preprocessors.payment_pattern_aggregator import PaymentPatternsAggregatorV2
from model_engine.assets.utils import load_asset

In [9]:
import model_engine
print('Loaded from:', model_engine.__file__)

Loaded from: /home/jag/model-engine/model_engine/__init__.py


In [11]:
import feature_engine_parts
print('Loaded from:', feature_engine_parts.__file__)

Loaded from: /home/jag/feature-engine-parts-processor-change/feature_engine_parts/__init__.py


# Confirming we have the updated PaymentPatternsAggregatorV2

In [12]:
import inspect

print(inspect.getsource(PaymentPatternsAggregatorV2._construct_trended_features))



    def _construct_trended_features(self, data, new_ppt, missing_data_chars):
        for month_range in self.month_ranges:
            trimmed = new_ppt.str[:month_range]
            # Denominator built on the OBSERVED string length, not the nominal
            # window, so bureaus that produce payment-pattern strings shorter
            # than the window (Experian, TransUnion) aren't penalized by an
            # inflated `month_range`. `all_month_range=False` then makes
            # _get_effective_month_range subtract every missing-data char in
            # the window -- not just trailing ones -- which is the spec-correct
            # definition of "observed months" for a trended rate denominator.
            effective_month_count = self._get_effective_month_range(
                trimmed,
                month_range=trimmed.str.len(),
                missing_data_chars=missing_data_chars,
                all_month_range=False,
            )
            for rate, values in self._

In [13]:
import inspect

print(inspect.getsource(PaymentPatternsAggregatorV2._get_count))



    def _get_count(self, trimmed, values):
        # pandas Series.str.count() parses `pat` as a REGEX. Bureau
        # missing-data chars include `*` (Equifax), which is a regex
        # quantifier with no preceding atom -- `re.compile('*')` raises
        # "nothing to repeat". Escape every value so we count it literally,
        # not as a pattern. Plain alphanumerics (e.g. '0'..'9') are
        # unchanged by re.escape so existing call sites keep working.
        if len(values) == 1:
            return trimmed.str.count(re.escape(values[0]))
        else:
            pattern = "|".join(re.escape(v) for v in values)
            return trimmed.str.count(pattern)



# Confirming our assets have the missing_data_chars

In [14]:
assets = {
  'equifax':    load_asset('equifax/cms_6/fe2/trade.json'),
  'experian':   load_asset('experian/arf7/fe2/trade.json'),
  'transunion': load_asset('transunion/TU4R/fe2/trade.json'),
}

In [15]:
assets['equifax']['preprocess'][3]

{'type': 'PaymentPatternsAggregatorV2',
 'params': {'report_date': 'rptDate',
  'missing_data_chars': ['*'],
  'payment_patterns': {'patterns': ['RATE_STATUS_CODE',
    'PAYMENT_HISTORY_1_24',
    'PAYMENT_HISTORY_25_36',
    'PAYMENT_HISTORY_37_48'],
   'rate': {'paid_as_agreed': ['0', '1'],
    'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ90+': ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ120+': ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'CO': ['6', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ30': ['2'],
    'DQ60': ['3'],
    'DQ90': ['4'],
    'DQ120': ['5']},
   'trim': 48,
   'placeholder': '/',
   'keep': ['DQ30+',
    'DQ60+',
    'DQ90+',
    'DQ120+',
    'CO',
    'DQ30',
    'DQ60',
    'DQ90']}},
 'notes': "missing_data_chars source: Equifax STS TotalView Programming Guide, p. 3-30 -- '*' means 'Rate/Status was not available for that month'. Equifax retain

In [16]:
assets['experian']['preprocess'][5]

{'type': 'PaymentPatternsAggregatorV2',
 'params': {'report_date': 'rptDate',
  'missing_data_chars': ['-'],
  'payment_patterns': {'patterns': ['PAYMENT_PROFILE'],
   'rate': {'paid_as_agreed': ['0', 'C'],
    'DQ30+': ['1',
     '2',
     '3',
     '4',
     '5',
     '6',
     '7',
     '8',
     '9',
     'G',
     'H',
     'J',
     'K',
     'L'],
    'DQ60+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'H', 'J', 'K', 'L'],
    'DQ90+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'H', 'J', 'K', 'L'],
    'DQ120+': ['4', '5', '6', '7', '8', '9', 'G', 'H', 'J', 'K', 'L'],
    'CO': ['8', '9', 'G', 'H', 'J', 'K', 'L'],
    'DQ30': ['1'],
    'DQ60': ['2'],
    'DQ90': ['3'],
    'DQ120': ['4', '5', '6']},
   'trim': 48,
   'placeholder': '-',
   'keep': ['DQ30+',
    'DQ60+',
    'DQ90+',
    'DQ120+',
    'CO',
    'DQ30',
    'DQ60',
    'DQ90']}},
 'notes': "missing_data_chars source: Experian CIS Credit Report Cross Reference Guide (ARF/XML/JSON), p. 49 -- '-' means 'No history 

In [17]:
assets['transunion']['preprocess'][2]

{'type': 'DateDiffV2',
 'params': {'feature': 'lstPmtDate',
  'reference_feature': 'date_of_request',
  'new_feature': 'months_since_lstPmtDate'}}

# Chunked processing pipeline — NEW PaymentPatternsAggregatorV2

This notebook runs the production feature pipeline end-to-end against the **updated** `PaymentPatternsAggregatorV2` from the editable `feature-engine-parts-processor-change` checkout. Its output is the "new-method" side of the new-vs-old comparison.

**What's new in this aggregator (brief):** `_construct_trended_features` now accepts `missing_data_chars` and computes the `percent_<rate>_<window>_months` denominator from the OBSERVED payment-pattern length, subtracting BOTH `#` (pre-report-date filler) AND the bureau's missing-data char (`*` Equifax / `-` Experian / `X` TransUnion) wherever they appear in the trimmed window. Upstream silently treated those positions as observed paid-as-agreed months, inflating the denominator and biasing DQ-rate features toward zero. **See `README.md` for the full write-up and per-bureau spec citations.**

## Pipeline

Mapping (`MapperV2`) has already been done by `map_and_save_mapped_data.ipynb` — its output lives at `payment_processing_research_data/<bureau>/<split>/mapped/`. This notebook picks up from there.

The notebook is organized into three sections — **Train**, **Valid**, **Test** — and each section calls `process_split(split)`. Per (split, bureau) `process_split` runs two phases:

1. **Phase 1 — preprocess in chunks (by row):** walk `mapped/part-NNNNN.parquet` one file at a time. For each chunk run the bureau's full `asset['preprocess']` list (DateDiffs, Coalesces, the updated `PaymentPatternsAggregatorV2`, DynamicPlaceholders, BivariateComputes, Filters, RowDroppers). Save 1:1 to `normalized/part-NNNNN.parquet`. Memory bounded to ~one chunk.

2. **Phase 2 — aggregate in chunks (by ZEST_KEY):** load the ENTIRE normalized dataset for this (bureau, split). Hash `ZEST_KEY` into 100 buckets (`hash(ZEST_KEY) % 100`) so every tradeline for a given ZEST_KEY lands in exactly one bucket. For each bucket run the **shared** `AggregationEngine` driven by `aggregation/fe2/trade.json` (`FilterAggregationV2`) — the real per-ZEST_KEY groupby that collapses many tradeline rows into one row per ZEST_KEY. Save to `processed/part-NNN.parquet`.

## Outputs

```
payment_processing_research_data/<bureau>/<split>/
├── normalized/  part-00000.parquet … part-NNNNN.parquet   (Phase 1, 1:1 with mapped)
└── processed/   part-000.parquet  … part-099.parquet      (Phase 2, 100 ZEST_KEY-hash groups, 1 row per ZEST_KEY)
```

In [18]:
import gc
import warnings
from pathlib import Path

import pandas as pd

from model_engine.feature_engine_V2.listed_objects_engines import MapperV2, PreprocessorV2
from model_engine.feature_engine_V2.feature_engine    import AggregationEngine
from model_engine.assets.utils                         import load_asset

from configs import (
    EQUIFAX, EXPERIAN, TRANSUNION, DATA_DIR,
    unmapped_dir, normalized_dir, processed_dir,
)

warnings.filterwarnings('ignore')

BUREAUS   = [EQUIFAX, EXPERIAN, TRANSUNION]
N_BUCKETS = 100   # ZEST_KEY hash buckets for the aggregation phase.


# Per-bureau MapperV2 + PreprocessorV2. Together these two are exactly what
# `InputNormalizer` runs in production: MapperV2 over asset['mapping'] turns
# the raw Snowflake bureau columns into typed/converted features, and
# PreprocessorV2 over asset['preprocess'] then runs DateDiffs, Coalesces,
# PaymentPatternsAggregatorV2, DynamicPlaceholders, BivariateComputes,
# Filters, RowDroppers. We do BOTH inside process_split so this notebook
# reads directly from `unmapped/` (which has the data) and doesn't depend
# on map_and_save_mapped_data.ipynb having finished.
mappers       = {}
preprocessors = {}
for cfg in BUREAUS:
    a = assets[cfg['bureau']]
    mappers[cfg['bureau']]       = MapperV2(api=a['mapping'])
    preprocessors[cfg['bureau']] = PreprocessorV2(api=a['preprocess'])
    print(f"{cfg['bureau']:11s}  MapperV2 ({len(a['mapping'])} steps)  +  "
          f"PreprocessorV2 ({len(a['preprocess'])} steps)")


# Shared AggregationEngine across ALL bureaus -- single
# `aggregation/fe2/trade.json` asset, same one used in production. This is
# the FilterAggregationV2 step that actually aggregates tradeline rows per
# ZEST_KEY into one output row per ZEST_KEY (group-by ZEST_KEY semantics).
# Because aggregation is per-ZEST_KEY, every tradeline for a given ZEST_KEY
# must land in the SAME input batch -- enforced by hash-bucketing in phase 2.
agg_asset = load_asset('aggregation/fe2/trade.json')
agg_eng   = AggregationEngine(asset=agg_asset, table_name='trade')
print(f"\nAggregationEngine built  key={agg_asset['aggregator']['key']}  "
      f"feature_groups={len(agg_asset['aggregator']['feature_groups'])}  "
      f"aggregations={len(agg_asset['aggregator']['aggregations'])}")


def clear_dir(d):
    """Delete every file under d, then ensure the directory exists empty."""
    d = Path(d)
    if d.exists():
        for f in d.glob('*'):
            if f.is_file():
                f.unlink()
    d.mkdir(parents=True, exist_ok=True)


def process_split(split):
    """Run the full two-phase pipeline for one split across all three bureaus.

    PHASE 1 -- map + preprocess (chunked by ROW)
        Per bureau, walk unmapped/part-NNNNN.parquet one at a time. For
        each chunk run:
          (a) MapperV2  over asset['mapping']    -- raw bureau cols -> typed features
          (b) PreprocessorV2 over asset['preprocess'] -- DateDiffs, Coalesces,
              PaymentPatternsAggregatorV2, DynamicPlaceholders, BivariateComputes,
              Filters, RowDroppers
        Save to normalized/part-NNNNN.parquet (1:1 with input). All these
        steps are row-independent, so chunking by row is safe.

    PHASE 2 -- aggregate (chunked by ZEST_KEY group)
        Load the ENTIRE normalized dataset for this (bureau, split). Hash
        ZEST_KEY into N_BUCKETS groups so every tradeline for a given
        ZEST_KEY lands in the same group. Run the SHARED AggregationEngine
        (`FilterAggregationV2`) on each group -- this is the real per-
        ZEST_KEY aggregation that collapses multiple tradeline rows into
        one row per ZEST_KEY. Save to processed/part-NNN.parquet.

    Input / Output layout (paths from configs.py):
        in :  payment_processing_research_data/<bureau>/<split>/unmapped/part-NNNNN.parquet
        out:  payment_processing_research_data/<bureau>/<split>/normalized/part-NNNNN.parquet
              payment_processing_research_data/<bureau>/<split>/processed/part-NNN.parquet
    """
    print(f'\n##### SPLIT = {split} #####')
    for cfg in BUREAUS:
        bureau   = cfg['bureau']
        in_dir   = Path(unmapped_dir(cfg, split))
        norm_dir = Path(normalized_dir(cfg, split))
        proc_dir = Path(processed_dir(cfg, split))

        print(f'\n=== {bureau}/{split} ===')
        parts = sorted(in_dir.glob('part-*.parquet')) if in_dir.exists() else []
        if not parts:
            print(f'[{bureau}/{split}]   MISSING unmapped chunks at {in_dir} '
                  f'-- run save_unmapped_data.ipynb first')
            continue

        clear_dir(norm_dir)
        clear_dir(proc_dir)

        mapper       = mappers[bureau]
        preprocessor = preprocessors[bureau]

        # ============ PHASE 1: map + full preprocess, chunked by row ============
        print(f'[{bureau}/{split}]  phase 1: map + preprocess ({len(parts)} chunks) -> {norm_dir}')
        for i, in_path in enumerate(parts):
            chunk      = pd.read_parquet(in_path)
            mapped     = mapper.transform(chunk)
            normalized = preprocessor.transform(mapped)
            normalized.to_parquet(norm_dir / f'part-{i:05d}.parquet', index=False)
            if (i + 1) % 50 == 0 or i == len(parts) - 1:
                print(f'  normalized {i + 1}/{len(parts)} chunks')
            del chunk, mapped, normalized
            gc.collect()

        # ============ PHASE 2: aggregate, chunked by ZEST_KEY group ============
        print(f'[{bureau}/{split}]  phase 2: load full normalized, split into '
              f'{N_BUCKETS} ZEST_KEY groups -> {proc_dir}')
        df = pd.read_parquet(norm_dir)
        print(f'  loaded {len(df):,} normalized rows')

        # Deterministic hash partition: same ZEST_KEY -> same bucket every time.
        df['_bucket'] = (
            pd.util.hash_pandas_object(df['ZEST_KEY'], index=False) % N_BUCKETS
        ).astype('int16')

        total_rows = 0
        for b in range(N_BUCKETS):
            sub = df[df['_bucket'] == b].drop(columns='_bucket')
            if not len(sub):
                continue
            # AggregationEngine collapses multiple rows per ZEST_KEY into one.
            processed = agg_eng.transform(sub)
            processed.to_parquet(proc_dir / f'part-{b:03d}.parquet', index=False)
            total_rows += len(processed)
            if (b + 1) % 10 == 0 or b == N_BUCKETS - 1:
                print(f'  aggregated group {b + 1}/{N_BUCKETS}  '
                      f'({total_rows:,} ZEST_KEYs so far)')
            del sub, processed
            gc.collect()

        del df
        gc.collect()
        print(f'[{bureau}/{split}]  done: {len(parts)} normalized chunks, '
              f'{N_BUCKETS} processed groups, {total_rows:,} ZEST_KEYs')
        print(f'   normalized -> {norm_dir}')
        print(f'   processed  -> {proc_dir}')

equifax      MapperV2 (30 steps)  +  PreprocessorV2 (100 steps)
experian     MapperV2 (35 steps)  +  PreprocessorV2 (107 steps)
transunion   MapperV2 (28 steps)  +  PreprocessorV2 (97 steps)

AggregationEngine built  key=ZEST_KEY  feature_groups=31  aggregations=40


## Train

In [ ]:
process_split('train')


##### SPLIT = train #####

=== equifax/train ===
[equifax/train]  phase 1: map + preprocess (582 chunks) -> /home/jag/payment-processor-research/payment_processing_research_data/equifax/train/normalized


## Valid

In [ ]:
process_split('valid')

## Test

In [ ]:
process_split('test')